<a href="https://colab.research.google.com/github/dsaoulis/Thesis/blob/main/ML(RF%2C_XB%2C_SVM%2C_Mixed_Model)_with_LP_Group_Weights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --------------------------------------------------------------------------------
# Classification Model
# Random Forest + XGBoost + SVM (PARALLEL COMPARISON)
# LP GROUP WEIGHTS
# --------------------------------------------------------------------------------

# --------------------------------------------------------------------------------
# 1. INSTALL REQUIRED PACKAGES
# --------------------------------------------------------------------------------
import subprocess
import sys

def install_packages():
    """Install required packages"""
    packages = [
        'pandas', 'numpy', 'matplotlib', 'seaborn',
        'scikit-learn', 'statsmodels', 'scipy', 'openpyxl', 'xgboost'
    ]
    for package in packages:
        try:
            __import__(package.replace('-', '_'))
            print(f"✓ {package} already installed")
        except ImportError:
            print(f"↓ Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
            print(f"✓ {package} installed")

print("-" * 80)
print("1. ΕΓΚΑΤΑΣΤΑΣΗ ΠΑΚΕΤΩΝ")
print("-" * 80)
install_packages()

# --------------------------------------------------------------------------------
# 2. IMPORT LIBRARIES
# --------------------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 10

print("\n✓ Όλα τα πακέτα εισήχθησαν με επιτυχία")

# --------------------------------------------------------------------------------
# 3. LOAD DATA FROM EXCEL
# --------------------------------------------------------------------------------
print("\n\n" + "-" * 80)
print("2. ΦΟΡΤΩΣΗ ΔΕΔΟΜΕΝΩΝ ΑΠΟ ΤΟ EXCEL")
print("-" * 80)

# Φόρτωση EXCEL από το dropbox μου
dropbox_url = "https://www.dropbox.com/scl/fi/ridnctbeg4tdjsr16cbpl/NORMALIZED-INDICATORS.xlsx?rlkey=96fvv6xnnwampnsm6gb1055k7&st=ujqqamyv&dl=1"
df = pd.read_excel(dropbox_url, sheet_name='ΣΥΝΟΛΙΚΑ 3')

print(f"✓ Φορτωμένα Δεδομένα: {df.shape[0]} σειρές, {df.shape[1]} στήλες")

# --------------------------------------------------------------------------------
# 4. DATA PREPROCESSING
# --------------------------------------------------------------------------------
print("\n\n" + "-" * 80)
print("3. ΠΡΟΕΠΕΞΕΡΓΑΣΙΑ ΔΕΔΟΜΕΝΩΝ")
print("-" * 80)

# Ορισμός Δεικτών και Rating
predictors = ['TAT', 'OIG', 'ROA', 'LEV', 'LIQ', 'ESG_Score']
target = 'S&P Credit Rating'
valid_ratings = ['AAA', 'AA+', 'AA', 'AA-', 'A+', 'A', 'A-', 'BBB+', 'BBB', 'BBB-', 'BB+', 'BB', 'BB-']

# Έλεγχος εγκυρότητάς τους
df_work = df[[*predictors, target]].copy().dropna().drop_duplicates()
df_work.columns = ['TAT', 'OIG', 'ROA', 'LEV', 'LIQ', 'ESG', 'Rating']
df_work['Rating'] = df_work['Rating'].astype(str).str.strip()
df_work = df_work[df_work['Rating'].isin(valid_ratings)].copy()

# Δημιουργία των 4 Groups
group_mapping = {
    'AAA': 'Group 1 (AAA/AA)', 'AA+': 'Group 1 (AAA/AA)', 'AA': 'Group 1 (AAA/AA)', 'AA-': 'Group 1 (AAA/AA)',
    'A+': 'Group 2 (A)', 'A': 'Group 2 (A)', 'A-': 'Group 2 (A)',
    'BBB+': 'Group 3 (BBB)', 'BBB': 'Group 3 (BBB)', 'BBB-': 'Group 3 (BBB)',
    'BB+': 'Group 4 (BB)', 'BB': 'Group 4 (BB)', 'BB-': 'Group 4 (BB)',
}

# Ιεράρχηση Ratings και Groups
ordered_ratings = ['AAA', 'AA+', 'AA', 'AA-', 'A+', 'A', 'A-', 'BBB+', 'BBB', 'BBB-', 'BB+', 'BB', 'BB-']
group_order = ['Group 1 (AAA/AA)', 'Group 2 (A)', 'Group 3 (BBB)', 'Group 4 (BB)']

# Μετατροπή των Ratings σε αυστηρή ιεραρχική κλίμακα
df_work['Rating_Cat'] = pd.Categorical(df_work['Rating'], categories=ordered_ratings, ordered=True)
df_work['Target_Code'] = df_work['Rating_Cat'].cat.codes

# Αφαιρούμε όσα δεν αντιστοιχούν σε γνωστό rating
df_work = df_work[df_work['Target_Code'] >= 0]

# Κατηγοριοποίηση σε ομάδες (για Classification)
df_work['Group'] = df_work['Rating'].map(group_mapping)
df_work = df_work[df_work['Group'].notna()]

# Κωδικοποίηση των ομάδων
label_encoder = LabelEncoder()
df_work['Group_Code'] = label_encoder.fit_transform(df_work['Group'])

print(f"\nΔιανομή Group:\n{df_work['Group'].value_counts()}")
print(f"\nΔιανομή Rating:\n{df_work['Rating'].value_counts()}")

print(f"\n✓ Dataset μέγεθος: {df_work.shape[0]} εταιρείες")

# --------------------------------------------------------------------------------
# 5. EXPLORATORY DATA ANALYSIS (EDA)
# --------------------------------------------------------------------------------
print("\n\n" + "-" * 80)
print("4. EXPLORATORY DATA ANALYSIS (EDA)")
print("-" * 80)

# Ανάλυση συσχέτισης με Spearman
corr_matrix = df_work[['TAT', 'OIG', 'ROA', 'LEV', 'LIQ', 'ESG', 'Group_Code']].corr(method='spearman')
print("Πίνακας Συσχέτισης Spearman:")
print(corr_matrix['Group_Code'].to_string())

# --------------------------------------------------------------------------------
# 6. TRAIN-TEST SPLIT & LP WEIGHT INTEGRATION
# --------------------------------------------------------------------------------
print("\n\n" + "-" * 80)
print("5. TRAIN-TEST SPLIT & ΕΝΣΩΜΑΤΩΣΗ LP ΒΑΡΩΝ")
print("-" * 80)

X = df_work[['TAT', 'OIG', 'ROA', 'LEV', 'LIQ', 'ESG']].copy()
y = df_work['Group_Code'].copy()

y_groups = df_work['Group'].copy()

X_train, X_test, y_train, y_test, y_train_groups, y_test_groups = train_test_split(
    X, y, y_groups, test_size=0.2, random_state=42, stratify=y
)

print(f"✓ Training set: {X_train.shape[0]} εταιρείες")
print(f"✓ Test set: {X_test.shape[0]} εταιρείες")

# Ενσωμάτωση βαρών από Linear Programming
lp_weights_dict = {
    'Group 1 (AAA/AA)': 2.6302,
    'Group 2 (A)': 0.9817,
    'Group 3 (BBB)': 0.5532,
    'Group 4 (BB)': 2.6302
}

sample_weights_train = y_train_groups.map(lp_weights_dict).values

print("\nΕνσωμάτωση βαρων στα Groups (Linear Programming):")
print("✓ Τα βέλτιστα βάρη εφαρμόστηκαν επιτυχώς σε κάθε δείγμα του training set.")

# --------------------------------------------------------------------------------
# 7. RANDOM FOREST CLASSIFIER
# --------------------------------------------------------------------------------
print("\n\n" + "-" * 80)
print("6. ΠΡΟΣΑΡΜΟΓΗ RANDOM FOREST ΤΑΞΙΝΟΜΗΤΗ")
print("-" * 80)

rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4]
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

rf_random = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_grid,
    n_iter=15,
    cv=3,
    scoring='f1_weighted',
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Εκπαίδευση με τα LP Weights μέσα στο Grid Search
rf_random.fit(X_train, y_train, sample_weight=sample_weights_train)
rf_model = rf_random.best_estimator_

print(f"\n✓ Βρέθηκαν οι βέλτιστες παράμετροι: {rf_random.best_params_}")
print("✓ Random Forest Ταξινομητής συντονίστηκε και εκπαιδεύτηκε επιτυχώς με LP βάρη")

# Feature importance
rf_feature_importance = pd.DataFrame({
    'Feature': ['TAT', 'OIG', 'ROA', 'LEV', 'LIQ', 'ESG'],
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nRandom Forest - Βαρύτητα Δεικτών:")
print(rf_feature_importance.to_string(index=False))

rf_pred_train = rf_model.predict(X_train)
rf_pred_test = rf_model.predict(X_test)
rf_pred_proba_test = rf_model.predict_proba(X_test)

# --------------------------------------------------------------------------------
# 8. XGBOOST CLASSIFIER (WITH HYPERPARAMETER TUNING & REGULARIZATION)
# --------------------------------------------------------------------------------
print("\n\n" + "-" * 80)
print("7. ΠΡΟΣΑΡΜΟΓΗ XGBOOST ΤΑΞΙΝΟΜΗΤΗ")
print("-" * 80)

xgb_param_grid = {
    'n_estimators': [100, 150, 200],
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01],
    'gamma': [0.1, 0.5, 1.0, 2.0],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.1, 1.0],
    'reg_lambda': [1.0, 5.0, 10.0]
}

xgb_base = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(group_order),
    random_state=42,
    verbosity=0,
    n_jobs=-1
)

# Στήσιμο του RandomizedSearchCV
xgb_random = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_grid,
    n_iter=15,
    cv=3,
    scoring='f1_weighted',
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Εκπαίδευση με ενσωμάτωση των LP Weights στο Grid Search
xgb_random.fit(X_train, y_train, sample_weight=sample_weights_train)
xgb_model = xgb_random.best_estimator_

print(f"\n✓ Βρέθηκαν οι βέλτιστες παράμετροι XGBoost:\n{xgb_random.best_params_}")
print("✓ XGBoost Ταξινομητής συντονίστηκε και εκπαιδεύτηκε επιτυχώς με LP βάρη")

# Feature importance
xgb_feature_importance = pd.DataFrame({
    'Feature': ['TAT', 'OIG', 'ROA', 'LEV', 'LIQ', 'ESG'],
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nXGBoost - Βαρύτητα Δεικτών:")
print(xgb_feature_importance.to_string(index=False))

xgb_pred_train = xgb_model.predict(X_train)
xgb_pred_test = xgb_model.predict(X_test)
xgb_pred_proba_test = xgb_model.predict_proba(X_test)


# --------------------------------------------------------------------------------
# 9. SUPPORT VECTOR MACHINES (SVM) CLASSIFIER
# --------------------------------------------------------------------------------
print("\n\n" + "-" * 80)
print("8. ΠΡΟΣΑΡΜΟΓΗ SUPPORT VECTOR MACHINES (SVM) ΤΑΞΙΝΟΜΗΤΗ")
print("-" * 80)

svm_param_grid = {
    'C': [0.1, 1.0, 1.5, 2.0, 2.5, 10.0, 50.0, 100.0, 500.0],
    'gamma': ['scale', 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 3.0],
    'kernel': ['rbf', 'linear']
}

svm_base = SVC(
    probability=True,
    random_state=42,
    verbose=False
)

# RandomizedSearchCV για τον SVM
svm_random = RandomizedSearchCV(
    estimator=svm_base,
    param_distributions=svm_param_grid,
    n_iter=15,
    cv=3,
    scoring='f1_weighted',
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Εκπαίδευση με ενσωμάτωση των LP Weights
svm_random.fit(X_train, y_train, sample_weight=sample_weights_train)
svm_model = svm_random.best_estimator_

print(f"\n✓ Βρέθηκαν οι βέλτιστες παράμετροι SVM:\n{svm_random.best_params_}")
print("✓ SVM Ταξινομητής συντονίστηκε και εκπαιδεύτηκε επιτυχώς με LP βάρη")

svm_pred_train = svm_model.predict(X_train)
svm_pred_test = svm_model.predict(X_test)
svm_pred_proba_test = svm_model.predict_proba(X_test)

# --------------------------------------------------------------------------------
# 10. VOTING CLASSIFIER (ENSEMBLE MODEL)
# --------------------------------------------------------------------------------
from sklearn.ensemble import VotingClassifier

print("\n\n" + "-" * 80)
print("9. ΠΡΟΣΑΡΜΟΓΗ ΤΑΞΙΝΟΜΗΤΗ ΨΗΦΟΦΟΡΙΑΣ (ΣΥΝΟΛΙΚΟ ΜΟΝΤΕΛΟ)")
print("-" * 80)

# Επιλογό μοντέλου με βάση τον μέσο όρο των πιθανοτήτων για καθε group
voting_clf = VotingClassifier(
    estimators=[
        ('rf', rf_model),
        ('xgb', xgb_model),
        ('svm', svm_model)
    ],
    voting='soft',
    weights=[3, 5, 1]
)

voting_clf.fit(X_train, y_train, sample_weight=sample_weights_train)
print("✓ Ταξινομητής Ψηφοφορίας (Συνολικο Μοντελο) εκπαιδευτηκε επιτυχώς")

voting_pred_train = voting_clf.predict(X_train)
voting_pred_test = voting_clf.predict(X_test)
voting_pred_proba_test = voting_clf.predict_proba(X_test)

# --------------------------------------------------------------------------------
# 11. MODEL PERFORMANCE COMPARISON - CLASSIFICATION METRICS
# --------------------------------------------------------------------------------
print("\n\n" + "-" * 80)
print("10. ΣΥΓΚΡΙΣΗ ΑΠΟΔΟΣΕΩΝ ΜΟΝΤΕΛΩΝ")
print("-" * 80)

def calculate_metrics(y_true, y_pred, y_pred_proba=None):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    try:
        if len(np.unique(y_true)) > 2 and y_pred_proba is not None:
            roc_auc = roc_auc_score(y_true, y_pred_proba, multi_class='ovr', average='weighted')
        elif y_pred_proba is not None:
            roc_auc = roc_auc_score(y_true, y_pred_proba[:, 1])
        else:
            roc_auc = np.nan
    except:
        roc_auc = np.nan

    return accuracy, precision, recall, f1, roc_auc

rf_acc_train, rf_prec_train, rf_rec_train, rf_f1_train, _ = calculate_metrics(y_train, rf_pred_train)
rf_acc_test, rf_prec_test, rf_rec_test, rf_f1_test, rf_roc_auc = calculate_metrics(y_test, rf_pred_test, rf_pred_proba_test)

xgb_acc_train, xgb_prec_train, xgb_rec_train, xgb_f1_train, _ = calculate_metrics(y_train, xgb_pred_train)
xgb_acc_test, xgb_prec_test, xgb_rec_test, xgb_f1_test, xgb_roc_auc = calculate_metrics(y_test, xgb_pred_test, xgb_pred_proba_test)

svm_acc_train, svm_prec_train, svm_rec_train, svm_f1_train, _ = calculate_metrics(y_train, svm_pred_train)
svm_acc_test, svm_prec_test, svm_rec_test, svm_f1_test, svm_roc_auc = calculate_metrics(y_test, svm_pred_test, svm_pred_proba_test)

voting_acc_train, voting_prec_train, voting_rec_train, voting_f1_train, _ = calculate_metrics(y_train, voting_pred_train)
voting_acc_test, voting_prec_test, voting_rec_test, voting_f1_test, voting_roc_auc = calculate_metrics(y_test, voting_pred_test, voting_pred_proba_test)

print("✓ Οι μετρήσεις υπολογίστηκαν επιτυχώς!")

# --------------------------------------------------------------------------------
# 12. COMPREHENSIVE COMPARISON TABLE
# --------------------------------------------------------------------------------
print("\n\n" + "-" * 80)
print("11. ΟΛΟΚΛΗΡΩΜΕΝΟ ΜΟΝΤΕΛΟ ΣΥΓΚΡΙΣΗΣ")
print("-" * 80)

comparison_data = {
    'Metric': [
        'Train Accuracy', 'Test Accuracy', 'Test Precision', 'Test Recall',
        'Test F1-Score', 'Test ROC-AUC'
    ],
    'Random Forest': [
        f"{rf_acc_train:.4f}", f"{rf_acc_test:.4f}", f"{rf_prec_test:.4f}",
        f"{rf_rec_test:.4f}", f"{rf_f1_test:.4f}", f"{rf_roc_auc:.4f}"
    ],
    'XGBoost': [
        f"{xgb_acc_train:.4f}", f"{xgb_acc_test:.4f}", f"{xgb_prec_test:.4f}",
        f"{xgb_rec_test:.4f}", f"{xgb_f1_test:.4f}", f"{xgb_roc_auc:.4f}"
    ],
    'SVM': [
        f"{svm_acc_train:.4f}", f"{svm_acc_test:.4f}", f"{svm_prec_test:.4f}",
        f"{svm_rec_test:.4f}", f"{svm_f1_test:.4f}", f"{svm_roc_auc:.4f}"
    ],
    'Voting Classifier': [
        f"{voting_acc_train:.4f}", f"{voting_acc_test:.4f}", f"{voting_prec_test:.4f}",
        f"{voting_rec_test:.4f}", f"{voting_f1_test:.4f}", f"{voting_roc_auc:.4f}"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Feature importance & ROC-AUC γραφήματα
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

  # 1. Feature importance σύγκριση  (RF vs XGBoost)
rf_feat = rf_feature_importance.copy()
rf_feat['Model'] = 'Random Forest'
xgb_feat = xgb_feature_importance.copy()
xgb_feat['Model'] = 'XGBoost'
feat_imp_df = pd.concat([rf_feat, xgb_feat])

plt.figure(figsize=(12, 6))
sns.barplot(data=feat_imp_df, x='Importance', y='Feature', hue='Model', palette='coolwarm')
plt.title('Σύγκριση Σπουδαιότητας Χαρακτηριστικών', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Σπουδαιότητα', fontsize=12, fontweight='bold')
plt.ylabel('Δείκτες', fontsize=12, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

  # 2. Προετοιμασία δεδομένων για πολυ-κλασικό ROC
y_test_bin = label_binarize(y_test, classes=[0, 1, 2, 3])
n_classes = y_test_bin.shape[1]

models_proba = [
    ("Random Forest", rf_pred_proba_test),
    ("XGBoost", xgb_pred_proba_test),
    ("SVM", svm_pred_proba_test),
    ("Voting Classifier", voting_pred_proba_test)
]

  # 3. 2x2 Grid ROC AUC καμπύλες ανα Group
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

macro_roc_data = []

for idx, (model_name, proba) in enumerate(models_proba):
    fpr = dict()
    tpr = dict()
    roc_auc = dict()

    # 3.1 Υπολογισμός ROC για κάθε Group ξεχωριστά
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        axes[idx].plot(fpr[i], tpr[i], linewidth=2,
                       label=f'{group_order[i]} (AUC = {roc_auc[i]:.2f})')

    axes[idx].plot([0, 1], [0, 1], 'k--', linestyle='--', alpha=0.5)
    axes[idx].set_xlim([0.0, 1.0])
    axes[idx].set_ylim([0.0, 1.05])
    axes[idx].set_xlabel('False Positive Rate (FPR)', fontsize=11)
    axes[idx].set_ylabel('True Positive Rate (TPR)', fontsize=11)
    axes[idx].set_title(f'ROC Curves: {model_name}', fontsize=13, fontweight='bold')
    axes[idx].legend(loc="lower right", frameon=True)
    axes[idx].grid(True, linestyle='--', alpha=0.5)

    # 3.2 Υπολογισμός και αποθήκευση του Macro-Average ROC
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    macro_auc = auc(all_fpr, mean_tpr)

    macro_roc_data.append((model_name, all_fpr, mean_tpr, macro_auc))

plt.tight_layout()
plt.show()

  # 4. Συνολικο διαγραμμα σύγκρισης ROC
plt.figure(figsize=(10, 7))
for model_name, all_fpr, mean_tpr, macro_auc in macro_roc_data:
    plt.plot(all_fpr, mean_tpr, linewidth=2.5,
             label=f'{model_name} (Macro-AUC = {macro_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (TPR)', fontsize=12, fontweight='bold')
plt.title('Συνολική Σύγκριση Καμπυλών ROC ', fontsize=14, fontweight='bold', pad=15)
plt.legend(loc="lower right", frameon=True, fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# --------------------------------------------------------------------------------
# 13. CONFUSION MATRICES VISUALIZATION
# --------------------------------------------------------------------------------
print("\n\n" + "-" * 80)
print("12. ΔΗΜΙΟΥΡΓΙΑ CONFUSION ΠΙΝΑΚΑ ΚΑΙ ΧΑΡΤΗ ΣΦΑΛΜΑΤΩΝ")
print("-" * 80)

unique_classes_in_test = np.unique(y_test)
group_labels_short = [group_order[i] for i in sorted(unique_classes_in_test)]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

def plot_cm(y_true, y_pred, ax, title, cmap):
    cm = confusion_matrix(y_true, y_pred, labels=unique_classes_in_test)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=group_labels_short,
                yticklabels=group_labels_short,
                ax=ax, cbar_kws={'label': 'Count'})
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Προβλεπόμενο Group')
    ax.set_ylabel('Πραγματικό Group')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')

# 1. Random Forest
plot_cm(y_test, rf_pred_test, axes[0], 'Random Forest Ταξινομητής\nConfusion Πίνακας', 'Blues')

# 2. XGBoost
plot_cm(y_test, xgb_pred_test, axes[1], 'XGBoost Ταξινομητής\nConfusion Πίνακας', 'Greens')

# 3. SVM
plot_cm(y_test, svm_pred_test, axes[2], 'SVM Ταξινομητής\nConfusion Πίνακας', 'Purples')

# 4. Voting Classifier
plot_cm(y_test, voting_pred_test, axes[3], 'Voting Classifier\nConfusion Πίνακας', 'Oranges')

plt.tight_layout()
plt.show()

# Misclassification heatmaps (ΑΝΑΛΥΣΗ ΣΦΑΛΜΑΤΩΝ)
fig_err, axes_err = plt.subplots(2, 2, figsize=(16, 14))
axes_err = axes_err.flatten()

def plot_err_matrix(y_true, y_pred, ax, title):
    cm = confusion_matrix(y_true, y_pred, labels=unique_classes_in_test)

    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_normalized = cm / row_sums

    # Mηδενισμός διαγωνίου: Κρατάμε μόνο τα λάθη ταξινόμησης
    np.fill_diagonal(cm_normalized, 0)

    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Reds',
                xticklabels=group_labels_short,
                yticklabels=group_labels_short,
                ax=ax, cbar_kws={'label': 'Misclassification Rate'})

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Προβλεπόμενο Group (Λανθασμένα)')
    ax.set_ylabel('Πραγματικό Group')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
    for label in ax.get_xticklabels():
        label.set_fontweight('bold')
    for label in ax.get_yticklabels():
        label.set_fontweight('bold')

# Σχεδίαση των Error Heatmaps για κάθε μοντέλο
plot_err_matrix(y_test, rf_pred_test, axes_err[0], 'Random Forest\nΧάρτης Σφαλμάτων')
plot_err_matrix(y_test, xgb_pred_test, axes_err[1], 'XGBoost\nΧάρτης Σφαλμάτων')
plot_err_matrix(y_test, svm_pred_test, axes_err[2], 'SVM\nΧάρτης Σφαλμάτων')
plot_err_matrix(y_test, voting_pred_test, axes_err[3], 'Voting Classifier\nΧάρτης Σφαλμάτων')

plt.tight_layout()
plt.show()

# --------------------------------------------------------------------------------
# 14. FULL ACCURACY SPECTRUM (EXACT & TOLERANCE METRICS)
# --------------------------------------------------------------------------------
print("\n\n" + "-" * 80)
print("13. ΛΕΠΤΟΜΕΡΗΣ ΑΚΡΙΒΕΙΑ (EXACT VS TOLERANCE)")
print("-" * 80)

def print_custom_accuracy(model_name, emoji, y_actual, y_pred, group_order):
    results = {g: {'total': 0, 'exact': 0, 'tol_1': 0} for g in group_order}
    accuracy_records = []

    for act_code, pred_code in zip(y_actual, y_pred):
        act_group = group_order[act_code]

        results[act_group]['total'] += 1

        if act_code == pred_code:
            results[act_group]['exact'] += 1
            results[act_group]['tol_1'] += 1
        elif abs(act_code - pred_code) <= 1:
            results[act_group]['tol_1'] += 1

    total_samples = len(y_actual)
    total_exact = sum(r['exact'] for r in results.values())
    total_tol_1 = sum(r['tol_1'] for r in results.values())

    acc_exact = (total_exact / total_samples) * 100 if total_samples > 0 else 0
    acc_tol_1 = (total_tol_1 / total_samples) * 100 if total_samples > 0 else 0

    print("\n" + emoji * 50)
    print(f"{model_name.upper()}")
    print("─" * 80)
    print(f"🎯 {model_name.upper()} - ΣΥΝΟΛΙΚΟ EXACT GROUP ACCURACY                  : {acc_exact:.2f}%")
    print(f"📈 {model_name.upper()} - ΣΥΝΟΛΙΚΟ ACCURACY ΜΟΝΤΕΛΟΥ  (Με ανοχή ±1 Group) : {acc_tol_1:.2f}%")
    print("─" * 80)

    header = f"{'Όνομα Group':<25} | {'Δείγμα':<6} | {'Exact Group':<14} | {'Tolerance ±1 Group':<20}"
    print(header)
    print("─" * 80)

    for group in group_order:
        data = results[group]
        total = data['total']
        if total > 0:
            exact = data['exact']
            tol_1 = data['tol_1']

            pct_exact = (exact / total) * 100
            pct_tol_1 = (tol_1 / total) * 100

            print(f"{group:<25} | {total:<6} | {pct_exact:5.1f}% ({exact:>2})    | {pct_tol_1:5.1f}% ({tol_1:>2})")

            accuracy_records.append({
                'Group': group,
                'Exact Group': pct_exact,
                'Tolerance (±1) Group': pct_tol_1
            })
    return accuracy_records

rf_records = print_custom_accuracy("Random Forest", "=", y_test, rf_pred_test, group_order)
xgb_records = print_custom_accuracy("XGBoost", "=", y_test, xgb_pred_test, group_order)
svm_records = print_custom_accuracy("SVM", "=", y_test, svm_pred_test, group_order)
voting_records = print_custom_accuracy("Voting Classifier", "=", y_test, voting_pred_test, group_order)

# ΔΙΑΓΡΑΜΜΑ ΑΚΡΙΒΕΙΑΣ (VOTING CLASSIFIER)
models_data = [
    ("Random Forest", rf_records),
    ("XGBoost", xgb_records),
    ("SVM", svm_records),
    ("Voting Classifier", voting_records)
]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, (model_name, records) in enumerate(models_data):
    if records:
        df_acc_plot = pd.DataFrame(records)
        df_acc_melted = df_acc_plot.melt(id_vars='Group', var_name='Metric', value_name='Accuracy %')

        sns.barplot(data=df_acc_melted, x='Group', y='Accuracy %', hue='Metric', palette=['#1f77b4', '#aec7e8'], ax=axes[idx])

        axes[idx].set_title(f'Διάγραμμα Ακρίβειας: {model_name}', fontsize=14, fontweight='bold', pad=15)
        axes[idx].set_xlabel('Groups', fontsize=12, labelpad=10)
        axes[idx].set_ylabel('Ποσοστό Ακρίβειας (%)', fontsize=12, labelpad=10)

        axes[idx].set_ylim(0, 115)
        if idx == 0:
            axes[idx].legend(loc='upper left', frameon=True)
        else:
            axes[idx].get_legend().remove()

        axes[idx].set_xticklabels(axes[idx].get_xticklabels(), fontweight='bold')
        axes[idx].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

--------------------------------------------------------------------------------
1. ΕΓΚΑΤΑΣΤΑΣΗ ΠΑΚΕΤΩΝ
--------------------------------------------------------------------------------
✓ pandas already installed
✓ numpy already installed
✓ matplotlib already installed
✓ seaborn already installed
↓ Installing scikit-learn...
✓ scikit-learn installed
✓ statsmodels already installed
✓ scipy already installed
✓ openpyxl already installed
✓ xgboost already installed

✓ Όλα τα πακέτα εισήχθησαν με επιτυχία


--------------------------------------------------------------------------------
2. ΦΟΡΤΩΣΗ ΔΕΔΟΜΕΝΩΝ ΑΠΟ ΤΟ EXCEL
--------------------------------------------------------------------------------
✓ Φορτωμένα Δεδομένα: 719 σειρές, 8 στήλες


--------------------------------------------------------------------------------
3. ΠΡΟΕΠΕΞΕΡΓΑΣΙΑ ΔΕΔΟΜΕΝΩΝ
--------------------------------------------------------------------------------

Διανομή Group:
Group
Group 3 (BBB)       378
Group 2 (A)    